In [1]:
import os
from pyspark.sql import SparkSession

# 1. Garante o uso do Java 17
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

# 2. Define os pacotes (JARs) que o Spark precisa baixar
pacotes = "io.delta:delta-spark_2.12:3.1.0,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0"

# 3. Configura o Builder híbrido (Delta + Iceberg)
builder = SparkSession.builder.appName("LakehouseHibrido") \
    .config("spark.jars.packages", pacotes) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension,org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "hadoop") \
    .config("spark.sql.catalog.iceberg.warehouse", "deltalake/iceberg") \
    .config("spark.sql.warehouse.dir", ".spark_warehouse")

# 4. Inicia a sessão 
spark = builder.getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark configurado com sucesso! Suporte a Delta e Iceberg ativado juntos.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/18 09:18:19 WARN Utils: Your hostname, saiti-HP-Elite-SFF-600-G9-Desktop-PC, resolves to a loopback address: 127.0.1.1; using 10.67.121.217 instead (on interface eno1)
26/03/18 09:18:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/silvio.ferreira/Projetos/estudo_lakehouse/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/silvio.ferreira/.ivy2.5.2/cache
The jars for the packages stored in: /home/silvio.ferreira/.ivy2.5.2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0a330143-154a-4678-916c-0533d0a0d021;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;

Spark configurado com sucesso! Suporte a Delta e Iceberg ativado juntos.


In [ ]:
comando_sql_do_livro = """
CREATE TABLE bronze_delta 
USING delta 
LOCATION 'lakehouse/bronze/events_delta/' 
AS SELECT * FROM parquet.`lakehouse/bronze/events_raw.parquet`;
"""

# 4. Executa o comando
spark.sql(comando_sql_do_livro)